In [1]:
import os
import random
import json
from PIL import Image, ImageDraw, ImageFont
from tqdm import tqdm
font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 48)
font_small = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 40)


In [2]:
def resize_keep_aspect(img, max_size):
    ratio = min(max_size[0] / img.width, max_size[1] / img.height)
    new_size = (int(img.width * ratio), int(img.height * ratio))
    return img.resize(new_size, Image.Resampling.LANCZOS), ratio

def create_reference_image(image_path, label_height=60):
    img = Image.open(image_path).convert('RGB')
    
    canvas = Image.new('RGB', (img.width, img.height + label_height), color='white')
    canvas.paste(img, (0, 0))
    
    draw = ImageDraw.Draw(canvas)
    bbox = draw.textbbox((0, 0), "REF", font=font)
    x = (img.width - (bbox[2] - bbox[0])) // 2
    y = img.height + (label_height - (bbox[3] - bbox[1])) // 2
    draw.text((x, y), "REF", fill='black', font=font)
    
    # Face bbox is entire image area (excluding REF label)
    face_bbox = [[0, 0], [img.width, img.height]]
    return canvas, face_bbox

In [3]:
def create_target_image(image_paths, output_size=(400, 400), label_height=60, padding=20):
    labels = ['A', 'B', 'C', 'D']
    cell_w, cell_h = output_size[0], output_size[1] + label_height
    
    canvas = Image.new('RGB', (2 * cell_w + 3 * padding, 2 * cell_h + 3 * padding), color='white')
    draw = ImageDraw.Draw(canvas)
    
    positions = [
        (padding, padding),
        (padding + cell_w + padding, padding),
        (padding, padding + cell_h + padding),
        (padding + cell_w + padding, padding + cell_h + padding)
    ]
    
    face_bboxes = []
    
    for img_path, label, pos in zip(image_paths, labels, positions):
        img = Image.open(img_path).convert('RGB')
        img, ratio = resize_keep_aspect(img, output_size)
        x_offset = pos[0] + (output_size[0] - img.width) // 2
        y_offset = pos[1] + (output_size[1] - img.height) // 2
        canvas.paste(img, (x_offset, y_offset))
        
        # Face bbox is the entire image area (excluding label)
        face_bbox = [[x_offset, y_offset], [x_offset + img.width, y_offset + img.height]]
        face_bboxes.append(face_bbox)
        
        bbox = draw.textbbox((0, 0), label, font=font_small)
        label_x = pos[0] + (cell_w - (bbox[2] - bbox[0])) // 2
        label_y = pos[1] + output_size[1] + (label_height - (bbox[3] - bbox[1])) // 2
        draw.text((label_x, label_y), label, fill='black', font=font_small)
    
    return canvas, face_bboxes

In [4]:
def collect_all_images(raw_image_folder, knowledge_probe_results=None):
    person_images = {}
    for person_id in os.listdir(raw_image_folder):
        if knowledge_probe_results is not None and person_id not in knowledge_probe_results:
            continue
        person_path = os.path.join(raw_image_folder, person_id)
        if os.path.isdir(person_path):
            images = [os.path.join(person_path, x) for x in os.listdir(person_path) if x.endswith('.jpg') or x.endswith('.png')]
            if len(images) >= 2:
                person_images[person_id] = images
    return person_images

def generate_hard_dataset(person_images, output_dir, num_samples=1000):
    os.makedirs(os.path.join(output_dir, 'ref'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'target'), exist_ok=True)
    
    annotations = []
    person_ids = list(person_images.keys())
    labels = ['A', 'B', 'C', 'D']
    
    for idx in tqdm(range(num_samples)):
        ref_person = random.choice(person_ids)
        ref_img_path, match_img_path = random.sample(person_images[ref_person], 2)
        
        distractor_persons = random.sample([p for p in person_ids if p != ref_person], 3)
        distractor_images = [random.choice(person_images[p]) for p in distractor_persons]
        
        all_target_images = [match_img_path] + distractor_images
        random.shuffle(all_target_images)
        correct_answer = labels[all_target_images.index(match_img_path)]
        
        ref_output_path = os.path.join(output_dir, 'ref', f'{idx:06d}_ref.jpg')
        ref_canvas, ref_bbox = create_reference_image(ref_img_path)
        ref_canvas.save(ref_output_path, quality=95)
        
        tgt_output_path = os.path.join(output_dir, 'target', f'{idx:06d}_target.jpg')
        tgt_canvas, tgt_bboxes = create_target_image(all_target_images)
        tgt_canvas.save(tgt_output_path, quality=95)
        
        annotations.append({
            "id": idx,
            "ref_image_path": ref_output_path,
            "tgt_image_path": tgt_output_path,
            "question": "Which person in the second image matches the person in the first image? Answer with A, B, C or D.",
            "options": ["A", "B", "C", "D"],
            "answer": correct_answer,
            "ref_coordinate": ref_bbox,
            "tgt_coordinate": tgt_bboxes,
            "metadata": {
                "ref_person_id": ref_person,
                "ref_original_image": ref_img_path,
                "match_original_image": match_img_path,
                "target_images_order": all_target_images,
                "distractor_persons": distractor_persons
            }
        })

    with open(os.path.join(output_dir, "annotations.json"), "w") as f:
        json.dump(annotations, f, indent=2)
    
    return annotations

In [5]:
QWEN_8B_KNOWS = ["Jackie_Chan", "Bruce_Lee", "Yao_Ming", "Steven_Yeun", "Jack_Ma", "BTS_Jin", "BTS_Jungkook", "BTS_Jimin", "BTS_Suga", "BTS_V"]
QWEN_4B_KNOWS = ["BTS_JHope", "Jackie_Chan", "BTS_RM", "Bruce_Lee", "Yao_Ming", "Steven_Yeun", "Jack_Ma", "BTS_Jimin", "BTS_Jin", "BTS_Suga", "BTS_V"]
QWEN_2B_KNOWS = ["Jackie_Chan","Bruce_Lee","Yao_Ming","Jack_Ma","BTS_Jin","Song_Heungmin","BTS_Jungkook","BTS_Jimin","BTS_Suga","BTS_V"]
GEMMA_12B_KNOWS = ["Jackie_Chan", "Steven_Yeun", "BTS_Jungkook", "BTS_Jimin", "BTS_V", "BTS_Suga", "BTS_RM", "BTS_Jin"]
GEMMA_4B_KNOWS = ["Jackie_Chan", "Steven_Yeun", "Bruce_Lee", "Jack_Ma", "BTS_Jungkook", "BTS_Jimin", "BTS_V", "BTS_RM", "BTS_Jin", "Masayoshi_Son"]


In [6]:
raw_image_folder = "RAW_IMAGES/famous_east_asian_males_nanobanana_cropped"
output_dir = "Qwen2B_FAMOUS_EAST_ASIAN_DATASET"
knowledge_probe_results = QWEN_2B_KNOWS 
person_images = collect_all_images(raw_image_folder, knowledge_probe_results)
print(f"Found {len(person_images)} persons")
random.seed(42)
annotations = generate_hard_dataset(person_images, output_dir, num_samples=1000)

Found 10 persons


100%|██████████| 1000/1000 [00:36<00:00, 27.51it/s]


In [7]:
raw_image_folder = "RAW_IMAGES/FLUXSynID_east_asian_cropped"
output_dir = "FLUXSynID_EAST_ASIAN_DATASET"
person_images = collect_all_images(raw_image_folder)
print(f"Found {len(person_images)} persons")
random.seed(42)
annotations = generate_hard_dataset(person_images, output_dir, num_samples=1000)

Found 14 persons


100%|██████████| 1000/1000 [00:24<00:00, 41.02it/s]
